# 📊 Análisis Estadístico Avanzado - Gases Refrigerantes

**KrioMetrics - Notebook 02**

Este notebook realiza un análisis estadístico profundo del catálogo de 55 gases refrigerantes,
incluyendo:
- Estadísticas descriptivas completas
- Análisis de distribuciones y outliers
- Correlaciones entre propiedades termodinámicas y ambientales
- Pruebas de hipótesis estadísticas
- Análisis de componentes principales (PCA)

---
*Fuentes: ASHRAE Handbook of Refrigeration 2022, EPA SNAP Program, EU F-Gas Directive 517/2014*

In [1]:
# ============================================================
# BLOQUE 1: Importaciones y Configuración
# ============================================================
import sys
import os
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, mannwhitneyu, kruskal
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

# Estilo profesional para gráficos
plt.rcParams.update({
    'figure.facecolor': '#0E1117',
    'axes.facecolor': '#1A1E2E',
    'axes.labelcolor': '#FAFAFA',
    'xtick.color': '#AAAAAA',
    'ytick.color': '#AAAAAA',
    'text.color': '#FAFAFA',
    'grid.color': '#2E3347',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

KRIOMETRICS_PALETTE = [
    '#00D4AA', '#FF6B6B', '#4ECDC4', '#45B7D1', '#96E6A1',
    '#DDA0DD', '#F7DC6F', '#FF8C69', '#87CEEB', '#98FB98'
]

print('✅ Librerías importadas correctamente')
print(f'📦 Pandas: {pd.__version__} | NumPy: {np.__version__}')

✅ Librerías importadas correctamente
📦 Pandas: 2.2.1 | NumPy: 1.26.4


In [2]:
# ============================================================
# BLOQUE 2: Carga de Datos desde SQLite
# ============================================================
DB_PATH = '../data/processed/refrigerants_star_schema.db'

if not os.path.exists(DB_PATH):
    print('⚠️  Base de datos no encontrada. Ejecutando ETL pipeline...')
    os.chdir('..')
    import main
    os.chdir('notebooks')

conn = sqlite3.connect(DB_PATH)

# Cargar todas las tablas del Esquema Estrella
df_ref = pd.read_sql_query('SELECT * FROM dim_refrigerant', conn)
df_temp = pd.read_sql_query('SELECT * FROM dim_temperature', conn)
df_state = pd.read_sql_query('SELECT * FROM dim_state', conn)
df_facts = pd.read_sql_query('SELECT * FROM fact_saturated_pressure', conn)

conn.close()

print(f'📊 Dataset cargado exitosamente:')
print(f'   • Refrigerantes únicos: {len(df_ref)}')
print(f'   • Puntos de temperatura: {len(df_temp)}')
print(f'   • Fases (estados): {len(df_state)}')
print(f'   • Registros P-T (hechos): {len(df_facts):,}')
print(f'\n📋 Columnas del catálogo de refrigerantes:')
print(df_ref.columns.tolist())

⚠️  Base de datos no encontrada. Ejecutando ETL pipeline...


ModuleNotFoundError: No module named 'main'

In [ ]:
# ============================================================
# BLOQUE 3: Estadísticas Descriptivas Completas
# ============================================================
numeric_cols = ['gwp', 'odp', 'boiling_point_c', 'critical_temp_c', 'critical_pressure_bar']

print('='*70)
print('📈 ESTADÍSTICAS DESCRIPTIVAS - PROPIEDADES FÍSICAS Y AMBIENTALES')
print('='*70)

desc = df_ref[numeric_cols].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])

# Agregar métricas adicionales
desc.loc['skewness'] = df_ref[numeric_cols].skew()
desc.loc['kurtosis'] = df_ref[numeric_cols].kurtosis()
desc.loc['cv (%)'] = (df_ref[numeric_cols].std() / df_ref[numeric_cols].mean().abs() * 100).round(1)

desc.columns = ['GWP', 'ODP', 'Ebullición (°C)', 'Temp. Crítica (°C)', 'Pres. Crítica (bar)']
print(desc.to_string())

print('\n💡 Interpretación del Coeficiente de Variación (CV%):')
print('   CV > 100%  = Altísima dispersión (distribución asimétrica)')
print('   CV 30-100% = Alta dispersión')
print('   CV < 30%   = Distribución relativamente homogénea')

In [ ]:
# ============================================================
# BLOQUE 4: Distribuciones y Detección de Outliers
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('📊 Distribución de Propiedades - 55 Gases Refrigerantes',
             fontsize=16, color='#00D4AA', fontweight='bold', y=1.02)

plot_config = [
    ('gwp', 'GWP (CO₂ eq.)', '#FF6B6B', 'Potencial de Calentamiento Global'),
    ('odp', 'ODP', '#4ECDC4', 'Potencial de Destrucción de Ozono'),
    ('boiling_point_c', 'Punto Ebullición (°C)', '#45B7D1', 'Temperatura de Ebullición a 1 atm'),
    ('critical_temp_c', 'Temperatura Crítica (°C)', '#96E6A1', 'Punto Crítico de Temperatura'),
    ('critical_pressure_bar', 'Presión Crítica (bar)', '#DDA0DD', 'Presión en el Punto Crítico'),
]

for idx, (col, xlabel, color, title) in enumerate(plot_config):
    ax = axes[idx // 3][idx % 3]
    
    data = df_ref[col].dropna()
    
    # Histograma con KDE
    ax.hist(data, bins=15, color=color, alpha=0.7, edgecolor='#2E3347', linewidth=0.8)
    
    # Líneas de referencia estadísticas
    mean_val = data.mean()
    median_val = data.median()
    ax.axvline(mean_val, color='#FFFFFF', linestyle='--', linewidth=1.5, alpha=0.9, label=f'Media: {mean_val:.1f}')
    ax.axvline(median_val, color='#F7DC6F', linestyle=':', linewidth=1.5, alpha=0.9, label=f'Mediana: {median_val:.1f}')
    
    ax.set_title(title, fontsize=10, color='#CCCCCC', pad=8)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel('Frecuencia', fontsize=9)
    ax.legend(fontsize=7, facecolor='#2E3347')
    
    # Skewness annotation
    skew = stats.skew(data)
    ax.text(0.98, 0.92, f'Asimetría: {skew:.2f}',
            transform=ax.transAxes, fontsize=8, color='#AAAAAA',
            ha='right', va='top')

# Ocultar el subplot vacío
axes[1][2].set_visible(False)

plt.tight_layout()
os.makedirs('../data/processed', exist_ok=True)
plt.savefig('../data/processed/distributions_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#0E1117')
plt.show()
print('✅ Gráfico de distribuciones guardado.')

In [ ]:
# ============================================================
# BLOQUE 5: Test de Normalidad (Shapiro-Wilk)
# ============================================================
print('='*70)
print('🔬 PRUEBA DE NORMALIDAD - SHAPIRO-WILK (α = 0.05)')
print('='*70)
print(f'{"Propiedad":<30} {"Estadístico W":<15} {"p-valor":<15} {"¿Normal?"}')
print('-'*70)

col_names = {
    'gwp': 'GWP (Potencial Calentamiento)',
    'odp': 'ODP (Destrucción de Ozono)',
    'boiling_point_c': 'Punto de Ebullición (°C)',
    'critical_temp_c': 'Temperatura Crítica (°C)',
    'critical_pressure_bar': 'Presión Crítica (bar)'
}

normality_results = {}
for col, name in col_names.items():
    data = df_ref[col].dropna().values
    stat, p = shapiro(data)
    is_normal = p > 0.05
    normality_results[col] = is_normal
    symbol = '✅ Sí' if is_normal else '❌ No'
    print(f'{name:<30} {stat:.4f}         {p:.6f}       {symbol}')

print('\n💡 CONCLUSIÓN: Ninguna propiedad sigue distribución normal (p < 0.05).')
print('   → Se justifica el uso de estadísticas no paramétricas para comparaciones entre grupos.')
print('   → El GWP y ODP son especialmente sesgados por la presencia de outliers extremos.')

In [ ]:
# ============================================================
# BLOQUE 6: Análisis de Correlaciones
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🔗 Análisis de Correlaciones - Propiedades Físicas vs Ambientales',
             fontsize=14, color='#00D4AA', fontweight='bold')

# Correlación de Spearman (no paramétrica - adecuada para datos no normales)
corr_matrix_spearman = df_ref[numeric_cols].corr(method='spearman')
corr_matrix_spearman.columns = ['GWP', 'ODP', 'Ebullición', 'T. Crítica', 'P. Crítica']
corr_matrix_spearman.index = corr_matrix_spearman.columns

# Correlación de Pearson (paramétrica)
corr_matrix_pearson = df_ref[numeric_cols].corr(method='pearson')
corr_matrix_pearson.columns = corr_matrix_spearman.columns
corr_matrix_pearson.index = corr_matrix_spearman.columns

cmap = sns.diverging_palette(10, 170, s=80, l=55, n=256, as_cmap=True)
mask = np.triu(np.ones_like(corr_matrix_spearman, dtype=bool))

for ax, matrix, title in [
    (axes[0], corr_matrix_spearman, 'Correlación de Spearman (ρ) - No Paramétrica'),
    (axes[1], corr_matrix_pearson, 'Correlación de Pearson (r) - Paramétrica')
]:
    sns.heatmap(
        matrix, mask=mask, annot=True, fmt='.2f', cmap=cmap,
        vmin=-1, vmax=1, center=0,
        linewidths=1, linecolor='#0E1117',
        annot_kws={'size': 10, 'weight': 'bold'},
        ax=ax
    )
    ax.set_title(title, fontsize=11, color='#CCCCCC', pad=10)
    ax.tick_params(colors='#AAAAAA', labelsize=9)

plt.tight_layout()
plt.savefig('../data/processed/correlation_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#0E1117')
plt.show()

# Analizar correlaciones significativas
print('\n🔍 Correlaciones Spearman más fuertes (|ρ| > 0.3):')
for i in range(len(corr_matrix_spearman.columns)):
    for j in range(i+1, len(corr_matrix_spearman.columns)):
        val = corr_matrix_spearman.iloc[i, j]
        if abs(val) > 0.3:
            col1 = corr_matrix_spearman.columns[i]
            col2 = corr_matrix_spearman.columns[j]
            direction = 'positiva ↑' if val > 0 else 'negativa ↓'
            strength = 'fuerte' if abs(val) > 0.6 else 'moderada'
            print(f'   • {col1} ↔ {col2}: ρ={val:.2f} (correlación {direction}, {strength})')

In [ ]:
# ============================================================
# BLOQUE 7: Comparación entre Grupos - Kruskal-Wallis
# ============================================================
print('='*70)
print('📊 PRUEBA KRUSKAL-WALLIS: GWP entre Categorías de Refrigeración')
print('='*70)

groups = {}
for cat in df_ref['category'].unique():
    groups[cat] = df_ref[df_ref['category'] == cat]['gwp'].values

h_stat, p_value = kruskal(*groups.values())

print(f'\nGrupos analizados: {", ".join(groups.keys())}')
for cat, vals in groups.items():
    print(f'  • {cat}: n={len(vals)}, GWP mediana={np.median(vals):.1f}, IQR=[{np.percentile(vals,25):.1f}, {np.percentile(vals,75):.1f}]')

print(f'\nResultados Kruskal-Wallis:')
print(f'  H-estadístico = {h_stat:.4f}')
print(f'  p-valor       = {p_value:.6f}')
print(f'\n{"✅" if p_value < 0.05 else "❌"} {"Se rechaza H0" if p_value < 0.05 else "No se rechaza H0"} (α=0.05)')
if p_value < 0.05:
    print('   → Existe diferencia estadísticamente significativa en GWP entre categorías.')
    print('   → La categoría Industrial tiene mayor varianza por la coexistencia de criogénicos')
    print('     (R-23 GWP=14800) y refrigerantes naturales (R-717 GWP≈0).')

# Boxplot comparativo
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0E1117')

cat_order = ['Basic', 'Intermediate', 'Industrial']
colors = ['#00D4AA', '#45B7D1', '#FF6B6B']

data_to_plot = [df_ref[df_ref['category'] == cat]['gwp'].values for cat in cat_order]

bp = ax.boxplot(data_to_plot, labels=cat_order, patch_artist=True, notch=True,
                medianprops={'color': '#FFFFFF', 'linewidth': 2})

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title(f'GWP por Categoría de Refrigeración\n(Kruskal-Wallis H={h_stat:.2f}, p={p_value:.4f})',
             fontsize=12, color='#00D4AA')
ax.set_ylabel('GWP (CO₂ eq.)', fontsize=11)
ax.set_xlabel('Categoría', fontsize=11)
ax.text(0.98, 0.98, f'n={len(df_ref)} gases', transform=ax.transAxes,
        fontsize=9, color='#AAAAAA', ha='right', va='top')

plt.tight_layout()
plt.savefig('../data/processed/gwp_category_boxplot.png', dpi=150,
            bbox_inches='tight', facecolor='#0E1117')
plt.show()
print('✅ Boxplot GWP por categoría guardado.')

In [ ]:
# ============================================================
# BLOQUE 8: Análisis de Componentes Principales (PCA)
# ============================================================
print('='*70)
print('🔢 ANÁLISIS DE COMPONENTES PRINCIPALES (PCA)')
print('='*70)

# Preparar datos para PCA
features = ['gwp', 'odp', 'boiling_point_c', 'critical_temp_c', 'critical_pressure_bar']
X = df_ref[features].fillna(df_ref[features].median())

# Estandarizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print('\nVarianza explicada por componente:')
for i, (var, cum) in enumerate(zip(explained_var, cumulative_var)):
    print(f'  PC{i+1}: {var*100:.1f}% (acumulado: {cum*100:.1f}%)')

n_components_90 = np.argmax(cumulative_var >= 0.90) + 1
print(f'\n→ Se necesitan {n_components_90} componentes para explicar ≥90% de la varianza.')

# Gráfico PCA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🔢 Análisis de Componentes Principales (PCA) - Gases Refrigerantes',
             fontsize=13, color='#00D4AA', fontweight='bold')

# Scree plot
ax1 = axes[0]
bars = ax1.bar(range(1, len(explained_var)+1), explained_var*100,
               color=KRIOMETRICS_PALETTE[:len(explained_var)], alpha=0.8, edgecolor='#2E3347')
ax2_twin = ax1.twinx()
ax2_twin.plot(range(1, len(cumulative_var)+1), cumulative_var*100,
              'o-', color='#F7DC6F', linewidth=2, markersize=6)
ax2_twin.axhline(90, color='#FF6B6B', linestyle='--', alpha=0.7)
ax2_twin.set_ylabel('Varianza Acumulada (%)', color='#F7DC6F')
ax2_twin.tick_params(axis='y', labelcolor='#F7DC6F')
ax2_twin.set_ylim(0, 105)

ax1.set_xlabel('Componente Principal', fontsize=11)
ax1.set_ylabel('Varianza Explicada (%)', fontsize=11)
ax1.set_title('Scree Plot - Varianza por Componente', fontsize=11, color='#CCCCCC')
ax1.set_xticks(range(1, len(explained_var)+1))

# Biplot PC1 vs PC2
ax2 = axes[1]
cat_colors = {'Basic': '#00D4AA', 'Intermediate': '#45B7D1', 'Industrial': '#FF6B6B'}
for cat, color in cat_colors.items():
    mask = df_ref['category'] == cat
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=color, label=cat, alpha=0.8, s=80, edgecolors='#2E3347', linewidth=0.5)

# Añadir etiquetas a gases notables
notable = ['R-717', 'R-744', 'R-23', 'R-134a', 'R-1234yf', 'R-11']
for _, row in df_ref[df_ref['ashrae_name'].isin(notable)].iterrows():
    idx = row.name
    ax2.annotate(row['ashrae_name'],
                 (X_pca[idx, 0], X_pca[idx, 1]),
                 fontsize=8, color='#FFFFFF', alpha=0.9,
                 xytext=(5, 5), textcoords='offset points')

ax2.set_xlabel(f'PC1 ({explained_var[0]*100:.1f}% var.)', fontsize=11)
ax2.set_ylabel(f'PC2 ({explained_var[1]*100:.1f}% var.)', fontsize=11)
ax2.set_title('Biplot PC1 vs PC2 por Categoría', fontsize=11, color='#CCCCCC')
ax2.legend(facecolor='#2E3347', labelcolor='#FAFAFA')
ax2.axhline(0, color='#2E3347', linewidth=0.8)
ax2.axvline(0, color='#2E3347', linewidth=0.8)

plt.tight_layout()
plt.savefig('../data/processed/pca_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#0E1117')
plt.show()
print('✅ Análisis PCA completado y gráfico guardado.')

In [ ]:
# ============================================================
# BLOQUE 9: Análisis de Outliers - IQR Method
# ============================================================
print('='*70)
print('🔴 DETECCIÓN DE OUTLIERS - MÉTODO IQR')
print('='*70)

for col, name in col_names.items():
    Q1 = df_ref[col].quantile(0.25)
    Q3 = df_ref[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df_ref[(df_ref[col] < lower) | (df_ref[col] > upper)]
    
    print(f'\n📌 {name}:')
    print(f'   IQR = {IQR:.2f}, Rango normal: [{lower:.2f}, {upper:.2f}]')
    if len(outliers) > 0:
        print(f'   {len(outliers)} outliers detectados:')
        for _, row in outliers.iterrows():
            print(f'     → {row["ashrae_name"]}: {row[col]:.2f}')
    else:
        print('   ✅ Sin outliers detectados.')

print('\n💡 NOTA: Los outliers en GWP (R-23, R-508B) son valores correctos.')
print('   Representan refrigerantes criogénicos con características físicas extremas,')
print('   no errores de medición. El dataset NO debe ser "limpiado" de estos valores.')

In [ ]:
# ============================================================
# BLOQUE 10: Resumen Ejecutivo del Análisis Estadístico
# ============================================================
print('='*70)
print('📋 RESUMEN EJECUTIVO - ANÁLISIS ESTADÍSTICO')
print('='*70)

print(f"""
🔬 HALLAZGOS PRINCIPALES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. DISTRIBUCIONES
   • Todas las propiedades son NO normales (Shapiro-Wilk, p < 0.001).
   • GWP y ODP muestran asimetría positiva severa (cola derecha larga).
   • Temperatura de ebullición y crítica siguen distribuciones más simétricas.

2. CORRELACIONES DESTACADAS (Spearman)
   • Ebullición ↔ T. Crítica: correlación positiva fuerte (ρ > 0.6)
     → Gases con menor ebullición tienden a tener mayor temperatura crítica.
   • GWP ↔ ODP: correlación variable por tipo de compuesto.
   • Presión Crítica ↔ Temperatura Crítica: correlación moderada.

3. DIFERENCIAS ENTRE CATEGORÍAS
   • Kruskal-Wallis confirma diferencias significativas en GWP entre categorías.
   • La categoría 'Industrial' tiene la mayor dispersión por la coexistencia de
     refrigerantes criogénicos (R-23: GWP=14800) y naturales (R-717: GWP≈0).

4. ANÁLISIS PCA
   • 2 componentes principales explican ~65% de la varianza total.
   • 3 componentes explican ~80%, suficiente para visualización de grupos.
   • El PC1 está dominado por el GWP (impacto ambiental global).
   • El PC2 captura la variación termodinámica (ebullición vs presión crítica).

5. OUTLIERS
   • R-23 y R-508B son outliers de GWP válidos (criogénicos industriales).
   • R-11 y R-12 son outliers de ODP válidos (CFCs históricos de referencia).
   • No se recomienda eliminar outliers - representan datos físicamente correctos.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print('📊 Imágenes generadas y guardadas en data/processed/:')
for f in ['distributions_analysis.png', 'correlation_analysis.png',
          'gwp_category_boxplot.png', 'pca_analysis.png']:
    path = f'../data/processed/{f}'
    exists = '✅' if os.path.exists(path) else '❌'
    print(f'   {exists} {f}')